# 2.1 — Procesamiento: calor extremo diario (Open-Meteo)

**Capa 2 del protocolo (clima, complemento diario).** Los resúmenes SMN son
mensuales y no permiten identificar olas de calor. Esta libreta revisa la
serie **diaria** de Open-Meteo (reanálisis) y su paso a métricas mensuales de
calor extremo. No tiene libreta de ingesta: `panel.load_openmeteo_daily()`
descarga la serie y la guarda en caché.

Metodología: `docs/docs/protocolo_eda.md` (sección 2.1, *Cómo medimos el
calor extremo*) y `docs/docs/heat_index.md`.

## Entradas

| Ruta | Contenido |
|------|-----------|
| `data/raw/openmeteo_hermosillo_daily_1991_2020.csv` | Tmax, Tmin y humedad relativa diarias (se descarga si no existe) |

## Salidas

- `data/interim/clima/openmeteo_hermosillo_mensual.csv` (opcional, para
  revisión): métricas mensuales de calor (`tmax_media_mes`,
  `n_olas_calor_p95`, `heat_index_max_mes`, …).

## Checklist

- [ ] Serie diaria completa 1991–2020 (sin días faltantes ni duplicados).
- [ ] Rangos físicos: `tmin_c ≤ tmax_c`, `0 ≤ humedad_rel_pct ≤ 100`.
- [ ] Umbrales P95 por mes calendario (línea base 1991–2020).
- [ ] Olas de calor: umbral fijo (≥35 °C), P95 adaptado y calor húmedo (heat index ≥40 °C).
- [ ] Agregación mensual (las olas se cuentan en el mes de su primer día).
- [ ] Contraste Open-Meteo vs. SMN (Tmax media mensual).

## Configuración e imports

In [ ]:
from pathlib import Path
import sys

ROOT_DIR = Path.cwd().parent.parent  # notebooks/<seccion>/ -> raiz del proyecto
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import pandas as pd
import matplotlib.pyplot as plt

from sonora_health_climate_eda.config import (
    HEAT_INDEX_DANGER_C,
    INTERIM_DATA_DIR,
    PANEL_END_YEAR,
    PANEL_START_YEAR,
)
from sonora_health_climate_eda.panel import (
    _openmeteo_daily_flags,  # privada: expone los flags diarios que usa el panel
    climate_hermosillo,
    load_openmeteo_daily,
    openmeteo_hermosillo_monthly,
)

OUT_DIR = INTERIM_DATA_DIR / "clima"
VARS_DIARIAS = ["tmax_c", "tmin_c", "humedad_rel_pct"]

## Completitud de la serie diaria

In [ ]:
diario = load_openmeteo_daily()
print(f"{len(diario):,} días, {diario['fecha'].min():%Y-%m-%d} a {diario['fecha'].max():%Y-%m-%d}")

calendario = pd.date_range(diario["fecha"].min(), diario["fecha"].max(), freq="D")
faltan = calendario.difference(diario["fecha"])
print(f"Días faltantes: {len(faltan)} | fechas duplicadas: {diario['fecha'].duplicated().sum()}")
print("\nNaN por variable:")
print(diario[VARS_DIARIAS].isna().sum())

## Rangos físicos

In [ ]:
tmin_mayor = diario[diario["tmin_c"] > diario["tmax_c"]]
hr_fuera = diario[(diario["humedad_rel_pct"] < 0) | (diario["humedad_rel_pct"] > 100)]
print(f"Días con Tmin > Tmax: {len(tmin_mayor)} | HR fuera de [0, 100]: {len(hr_fuera)}")
diario[VARS_DIARIAS].describe().T

## Umbrales P95 por mes calendario

Un día es *extremo* si Tmax **y** Tmin superan el P95 de su mes calendario
(línea base 1991–2020); una ola son ≥3 días seguidos así.

In [ ]:
flags = _openmeteo_daily_flags()
umbrales = (
    flags.groupby(flags["fecha"].dt.month)[["thr_tmax_p95", "thr_tmin_p95"]]
    .first()
    .rename_axis("mes")
    .round(1)
)
umbrales

## Días en ola de calor por definición (ventana del panel)

In [ ]:
DEF_OLAS = {
    "en_ola": "fija ≥35 °C",
    "en_ola_p95": "P95 adaptado",
    "en_ola_humedo": f"heat index ≥{HEAT_INDEX_DANGER_C:.0f} °C",
}
ventana = flags[flags["fecha"].dt.year.between(PANEL_START_YEAR, PANEL_END_YEAR)]
print("Días dentro de una ola de calor:")
print(ventana[list(DEF_OLAS)].sum().rename(DEF_OLAS))

frac_mes = ventana.groupby(ventana["fecha"].dt.month)[list(DEF_OLAS)].mean().rename(columns=DEF_OLAS)
ax = frac_mes.plot(kind="bar", figsize=(10, 4))
ax.set_xlabel("Mes calendario")
ax.set_ylabel("Fracción de días")
ax.set_title("Fracción de días en ola de calor por mes calendario")
plt.tight_layout()
plt.show()

## Agregación mensual

In [ ]:
om_mensual = openmeteo_hermosillo_monthly()
om_mensual = om_mensual[om_mensual["fecha_mes"].dt.year.between(PANEL_START_YEAR, PANEL_END_YEAR)]
print(f"{len(om_mensual)} meses en la ventana del panel")
om_mensual.head()

## Contraste Open-Meteo vs. SMN

El reanálisis es una celda de malla, no una estación: se espera un sesgo
sistemático. Lo importante es que ambas series se muevan juntas.

In [ ]:
smn = climate_hermosillo()[["fecha_mes", "temp_max_media_c"]]
comp = om_mensual[["fecha_mes", "tmax_media_mes"]].merge(smn, on="fecha_mes", how="inner").dropna()
print(f"Meses comparables: {len(comp)}")
print(f"Correlación: {comp['tmax_media_mes'].corr(comp['temp_max_media_c']):.3f}")
print(f"Sesgo medio (Open-Meteo − SMN): {(comp['tmax_media_mes'] - comp['temp_max_media_c']).mean():+.2f} °C")

ax = comp.plot.scatter(x="temp_max_media_c", y="tmax_media_mes", figsize=(5, 5))
lims = [comp[["temp_max_media_c", "tmax_media_mes"]].min().min(), comp[["temp_max_media_c", "tmax_media_mes"]].max().max()]
ax.plot(lims, lims, "--", color="gray")
ax.set_xlabel("SMN — Tmax media mensual (°C)")
ax.set_ylabel("Open-Meteo — Tmax media mensual (°C)")
plt.tight_layout()
plt.show()

## Guardar salida

In [ ]:
# TODO: guardar una vez validadas las métricas
# OUT_DIR.mkdir(parents=True, exist_ok=True)
# om_mensual.to_csv(OUT_DIR / "openmeteo_hermosillo_mensual.csv", index=False)

## Decisiones de procesamiento

Registrar cada decisión que cambie los datos y su justificación. Si la
decisión cambia el panel, implementarla en `sonora_health_climate_eda/panel.py`
(es lo que ejecuta `03_eda/3.0-integracion-espacial.ipynb`) y llevarla al
reporte final.

| Variable / periodo | Problema | Decisión | Justificación |
|--------------------|----------|----------|---------------|
| Ola de calor, umbral fijo (≥35 °C, ≥3 días) | En Hermosillo casi todo el verano lo cumple | Se conserva como referencia (`n_olas_calor`) | Convención SMN, pero discrimina poco |
| Ola de calor, P95 por mes calendario | Calor anómalo para la época | Métrica principal (`n_olas_calor_p95`) | Detecta olas fuera de temporada (p. ej. marzo) |
| Heat index (Rothfusz) | Sin ajustes del NWS para HR < 40 % | Se acepta la aproximación | Válida sobre todo en el verano húmedo (ago–oct); ver `heat_index.md` |
| _TODO_ | | | |